# Comparación TwinLiteNetPlus vs UFLDv2 (Colab)

1. **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.
2. Subí `kit_segmentadores.zip` a tu Google Drive (por ejemplo a `MyDrive/segmentadores/`) junto con tus videos en `MyDrive/segmentadores/videos/unlam` y `MyDrive/segmentadores/videos/bfmc`.
3. Ejecutá las celdas en orden.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/segmentadores'   # <- cambiá si lo pusiste en otro lado

In [ ]:
!rm -rf /content/kit && mkdir -p /content/kit
!unzip -q -o "$BASE/kit_segmentadores.zip" -d /content
%cd /content/kit
!ls

In [ ]:
!bash preparar.sh

### Prueba rápida (300 cuadros por video)

In [ ]:
TL = !find pesos/twinlite -name 'large.pth' | head -1
TL = TL[0]; print(TL)
!python comparar.py --videos "$BASE/videos/unlam" "$BASE/videos/bfmc" --salida resultados_prueba \
    --tl-weight {TL} --tl-config large --half \
    --ufld-weight pesos/ufld/culane_res18.pth --ufld-config culane_res18 --max-frames 300

### Corrida completa (y variantes para comparar)
Descomentá las que quieras. Cada una deja sus resultados en una carpeta distinta.

In [ ]:
!python comparar.py --videos "$BASE/videos/unlam" "$BASE/videos/bfmc" --salida resultados_culane \
    --tl-weight {TL} --tl-config large --half \
    --ufld-weight pesos/ufld/culane_res18.pth --ufld-config culane_res18

# UFLDv2 entrenado en TuSimple
# !python comparar.py --videos "$BASE/videos/unlam" "$BASE/videos/bfmc" --salida resultados_tusimple \
#     --tl-weight {TL} --tl-config large --half \
#     --ufld-weight pesos/ufld/tusimple_res18.pth --ufld-config tusimple_res18

# BFMC recortando el 20% superior de la imagen
# !python comparar.py --videos "$BASE/videos/bfmc" --salida resultados_bfmc_recorte \
#     --tl-weight {TL} --tl-config large --half \
#     --ufld-weight pesos/ufld/culane_res18.pth --ufld-config culane_res18 --recorte-sup 0.2

In [ ]:
from IPython.display import Markdown, Image, display
R = 'resultados_culane'
display(Markdown(open(f'{R}/resumen.md').read()))
display(Image(f'{R}/graficos_resumen.png'))
import glob
for p in sorted(glob.glob(f'{R}/*/*/centro_carril.png')): display(Image(p))
for p in sorted(glob.glob(f'{R}/*/*/capturas/*.png'))[:6]: display(Image(p, width=900))

### Guardar resultados en Drive

In [ ]:
!cp -r resultados_* "$BASE/" && echo 'Copiado a Drive'